# Réserve à okapis : audit d’un raster classé

**Cartomize · Version 1.0**

Durée indicative : 35 minutes.

## Objectifs

Inventorier les codes, préserver les valeurs ambiguës et produire une représentation catégorielle sans inventer la légende.

## Apport de Cartomize

Le même raster alimente le diagnostic, les comptes par classe, le tableau exporté et la carte avec nomenclature explicite.

Données : fichiers fournis dans `Tuto.zip`, importés localement. Consulter [les données et leurs limites](../DATA.md).

## Préparation

Installer l’environnement décrit dans le [guide de démarrage](../README.md), puis exécuter les cellules dans l’ordre. Chaque exécution utilise un nouveau dossier de résultats.

In [ ]:
from pathlib import Path
import sys, json
import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
import matplotlib.pyplot as plt
from IPython.display import display, Image
root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p/'course.py').is_file() or (p/'tutorials/course.py').is_file())
root = root if (root/'course.py').is_file() else root/'tutorials'
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
import cartomize as cm
from course import demo, run_directory, show_raster, show_rgb, gallery, CRS, CLASSES, ROLES

In [ ]:
out = run_directory("12_okapis_raster_classe")
from course import supplied_data
d = supplied_data()

## 1. Examiner le raster

Le fichier contient une bande uint8, décrite `LULC_2025`, sans NoData déclaré. Aucun dictionnaire de classes n’est fourni. Le nom « forêt » du fichier ne suffit pas à identifier les codes. Le SCR déclaré est EPSG:32634 ; la grande valeur d’abscisse exige une vérification auprès du producteur avant de publier des surfaces de référence.

In [ ]:
source=d['okapi']
display(cm.raster.inspect(source))
with rasterio.open(source) as src:
    print('Dimensions :',src.width,src.height,'| SCR :',src.crs,'| Résolution :',src.res)
    assert src.count==1

## 2. Compter toutes les classes par blocs

Le code 0 est conservé. Tant que sa signification n’est pas connue, il ne doit être ni masqué automatiquement ni assimilé à une classe forestière. Les hectares ci-dessous sont des surfaces planaires calculées dans le SCR déclaré, à confirmer après contrôle du géoréférencement.

In [ ]:
areas=cm.raster.class_areas(source,unit='ha')
areas.to_csv(out/'codes_surfaces_scr_declare.csv',index=False)
display(areas)
with rasterio.open(source) as src:
    assert areas['pixels'].sum()==src.width*src.height

## 3. Produire un aperçu catégoriel

Le voisin le plus proche conserve les codes. L’aperçu réduit sert à l’affichage uniquement ; le tableau précédent a été calculé sur tous les pixels à la résolution originale.

In [ ]:
from rasterio.enums import Resampling
from affine import Affine
preview=out/'apercu_codes.tif'
with rasterio.open(source) as src:
    ratio=min(1,1400/max(src.height,src.width));h=int(src.height*ratio);w=int(src.width*ratio)
    values=src.read(out_shape=(1,h,w),resampling=Resampling.nearest)
    profile=src.profile.copy();profile.update(width=w,height=h,transform=src.transform*Affine.scale(src.width/w,src.height/h),compress='lzw')
    with rasterio.open(preview,'w',**profile) as dst:dst.write(values)
codes=areas.iloc[:,0].tolist()
colors=['#efefeb','#194f36','#50845b','#8fac72','#b4b6a4','#bbaa84','#6f92a7','#a68a80']
classes={int(value):(f'Code {int(value)}',colors[i%len(colors)]) for i,value in enumerate(codes)}
carte=cm.Map(title='Réserve à okapis : codes du raster',subtitle='Fichier fourni, 2025 | Légende thématique non documentée',crs='EPSG:32634',credits='Fichier Tuto | Géoréférencement et nomenclature à confirmer')
carte.add_layer(preview,name='Code source',classes=classes)
carte.add_scale_bar(False).add_north_arrow(False)
carte.export(out/'codes_okapis.pdf',dpi=300)
fig=carte.render(dpi=140);gallery(fig,'12_okapis');plt.show()

## 4. Passer à une analyse thématique

Après confirmation des codes et du géoréférencement, renseigner `classes` pour l’affichage et une table `mapping` pour la reclassification. Une étude de changement exige un second raster daté, une nomenclature harmonisée, une grille commune et une validation. Le ZIP ne fournit pas cette seconde date.

## Exercice

Le code 0 occupe une grande partie de l’image. Est-ce suffisant pour le définir comme NoData ?

<details>
<summary>Éléments de correction</summary>

Non. Sa fréquence ou sa couleur ne démontre pas sa signification. Consulter les métadonnées du producteur ou une emprise valide documentée. Conserver le code dans l’audit tant que cette information manque.

</details>

## Reproduction

Les paramètres, la graine et les chemins sont explicites dans les cellules. Le répertoire `out` contient les produits de cette exécution. Adapter les sources, le système de coordonnées et les paramètres avant de transférer la méthode à une étude.